In [ ]:
from pathlib import Path
from figure_paths import DATA, RESULTS, FIGURES, setting

SCORE_DIR = setting('YANGTZE_SCORE_DIR', RESULTS / 'direct11')
RESULT_DIR = RESULTS
ERA5_DIR = DATA
FIGURE_DIR = FIGURES
DPI = 700

SOURCE = "auto"   # "tables" requires score tables. "netcdf" scores the prediction fields.
TABLE_FILES = []
TABLE_FILTERS = {}
BLOCK_HOURS = 168
RAW_PATHS = {}
RAW_SCHEMA = {}

RAW_SCORE_FILE = RESULTS / 'direct11' / 'FigS24_layer_scores.csv'


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


class ScoreTableError(ValueError):
    """The selected files do not establish one complete comparison."""


def _key(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).lower().replace('ω', 'omega').replace('Ω', 'omega'))


_MODEL_NAMES = {
    'Direct6': {'direct6', 'direct6h', 'direct6hour', 'd6', 'direct6hinput', 'direct6hmain', 'stage1direct6', 'direct6hstage1', 's1direct6', 'direct6hmainforecast', 'direct6single', 'direct6singlemodel', '6hdirect'},
    'Direct11': {'direct11', 'direct11h', 'direct11hour', 'd11', 'direct11hinput', 'direct11hmain', 'direct11single', 'direct11singlemodel', '11hdirect'},
    'Two-stage': {'twostage', 'twostagefinal', 'finaltwostage', '2stage', 'residualcorrected', 'residualcorrectedforecast', 'finalresidualcorrected', 'datransunettwostage', 'twostagedatransunet', 'stage2final', 'stage2finalforecast', 'stage2', 'twostage11h', 'twostage6plus6', 'finalstage2', 'residualcorrectedfinal', 'finaltwostagedatransunet'},
}
_METRIC_NAMES = {
    'R': {'r', 'corr', 'correlation', 'pearsonr', 'pearson', 'pearsoncorrelation', 'rmean', 'meanr', 'gridmeanr'},
    'RMSPE': {'rmspe', 'nrmse', 'rangenormalizedrmse', 'rangenormalisedrmse', 'nrmserange', 'nrmsemean', 'meannrmse', 'rmspemean', 'meanrmspe', 'gridmeannrmse'},
}
_DIM_NAMES = {
    'lead': {'lead', 'leadh', 'leadhour', 'leadhours', 'leadtime', 'leadtimeh', 'leadtimehours', 'leadtimehour', 'forecastleadh', 'forecastleadhours', 'forecastlead', 'forecasthour', 'horizon', 'horizonh'},
    'level': {'level', 'levelhpa', 'plev', 'pressure', 'pressurelevel', 'pressurehpa', 'pressurelevelhpa', 'isobaricinhpa'},
    'component': {'component', 'windcomponent', 'variable', 'var', 'targetvariable', 'field', 'windvar', 'windvariable', 'targetcomponent'},
    'model': {'model', 'modelname', 'modelid', 'configuration', 'config', 'method', 'architecture', 'experiment'},
    'metric': {'metric', 'metricname', 'scoremetric'},
    'value': {'value', 'score', 'estimate', 'pointestimate', 'metricvalue', 'meanvalue', 'spatialmean', 'gridmean', 'domainmean'},
}
_COMPONENTS = {
    'u': {'u', 'uwind', 'zonal', 'zonalwind', 'ucomponent', 'ucomponentofwind'},
    'v': {'v', 'vwind', 'meridional', 'meridionalwind', 'vcomponent', 'vcomponentofwind'},
    'omega': {'omega', 'w', 'vertical', 'verticalvelocity', 'pressureverticalvelocity', 'wcomponent'},
}
_IDENTIFIERS = ['model', 'lead', 'level', 'component']
_SCORE_METRICS = ['R', 'RMSPE']


def _which(value, names):
    key = _key(value)
    return next((name for name, aliases in names.items() if key in aliases), None)


def _coordinate(series, kind):
    if kind == 'lead':
        pattern = r'(?:lead\s*|h\s*)?([1-6])(?:\.0+)?\s*(?:h|hr|hrs|hours?)?'
    else:
        pattern = r'(850|925|1000)(?:\.0+)?\s*(?:hpa|mb|mbar)?'
    parsed = series.astype(str).str.strip().str.lower().str.extract('^' + pattern + '$', expand=False)
    return pd.to_numeric(parsed, errors='coerce')


def _metadata(frame, file):
    flags = {'mask', 'masktype', 'spatialsupport', 'terrainmask', 'maskmode', 'domainmask',
             'evaluationdomain', 'reference', 'truthsource', 'referencesource', 'timewindow',
             'targetwindow', 'testwindow', 'validtimestart', 'validtimeend', 'maskpolicy',
             'normalization', 'normalisation', 'support', 'maskname', 'gridmask', 'timesupport', 'truthversion', 'evaluationwindow', 'scoreprotocol', 'temporalaggregation', 'spatialaggregation'}
    meta = {}
    for col in frame.columns:
        k = _key(col)
        if k not in flags:
            continue
        vals = frame[col].dropna().map(_key).unique()
        if len(vals) > 1:
            raise ScoreTableError(f'{file.name}: {col} 包含多个取值，请指定 TABLE_FILTERS。')
        if not len(vals):
            continue
        val = vals[0]
        if (any(s in val for s in ('aboveground', 'abovesurface', 'terrainmasked', 'belowground', 'alllevelsmean'))
                or ('mask' in k and val in {'masked', 'true', '1', 'terrain', 'above'})):
            raise ScoreTableError(f'{file.name}: 检测到非原始全域评分 {col}={val}。')
        meta[k] = val
    for endpoint, names, expected in [
        ('开始', {'validtimestart','targettimestart','firsttargettime'}, '2019-03-15 02:00'),
        ('结束', {'validtimeend','targettimeend','lasttargettime'}, '2023-12-31 23:00'),
    ]:
        for col in (col for col in frame.columns if _key(col) in names):
            dates = pd.to_datetime(frame[col].dropna(), utc=True)
            if dates.nunique() != 1 or dates.iloc[0] != pd.Timestamp(expected, tz='UTC'):
                raise ScoreTableError(f'{file.name}: 共同目标时间的{endpoint}与正式窗口不符。')
    for col in frame.columns:
        if _key(col) in {'ncommontimes', 'ncommontargets', 'ntimes', 'ntarget', 'targetcount', 'ntimecommon', 'ntargets', 'ntimestamps', 'ncommonhours', 'ncommontesttimes', 'ncommontesthours', 'commonn', 'ntargethours', 'ncommontargethours', 'ncommon', 'ntargettimes', 'ntargettimestamps', 'ncommonvalidtimes', 'commontargethours'}:
            counts = pd.to_numeric(frame[col], errors='raise').dropna().unique()
            if len(counts) != 1 or counts[0] != 42070:
                raise ScoreTableError(f'{file.name}: {col} 不是预期的 42,070 个共同目标时次。')
    return meta


def _one_table(file, filters):
    df = pd.read_csv(file, encoding='utf-8-sig')
    for name, val in filters.items():
        if name not in df.columns:
            raise ScoreTableError(f'{file.name}: 没有筛选列 {name!r}。')
        df = df.loc[df[name].astype(str).str.casefold() == str(val).casefold()].copy()
    if df.empty:
        raise ScoreTableError(f'{file.name}: 没有保留记录。')
    for column in df.columns:
        k = _key(column)
        if k in {'statistic', 'stattype', 'stat', 'estimatetype', 'resulttype', 'scoretype'}:
            row_type = df[column].map(_key)
            drop = row_type.str.contains('quantile|bootstrap|delta|diff|gain|bound|interval|^ci|^lower|^upper|^stderr|^se$|^pvalue')
            df = df.loc[~drop]
    used = {}
    for dim, names in _DIM_NAMES.items():
        matches = [c for c in df if _key(c) in names]
        if len(matches) > 1 and dim != 'value':
            raise ScoreTableError(f'{file.name}: 同时存在 {dim} 候选列 {matches}。')
        if len(matches) == 1:
            used[matches[0]] = dim
    d = df.rename(columns=used)
    if 'lead' not in d:
        hits = set(re.findall(r'(?i)(?:lead[_-]?|(?:^|[_-])h)([1-6])(?=[_./-]|$)', file.as_posix()))
        if len(hits) == 1:
            d = d.assign(lead=int(hits.pop()))
    if not {'lead', 'level', 'component'}.issubset(d):
        raise ScoreTableError(f'{file.name}: 缺少分时效、分层或分量坐标；列名 {list(df)}。')
    d = d.copy()
    d['lead'] = _coordinate(d['lead'], 'lead')
    d['level'] = _coordinate(d['level'], 'level')
    d['component'] = d.component.map(lambda v: _which(v, _COMPONENTS))
    d = d.dropna(subset=['lead', 'level', 'component'])
    if d.empty:
        raise ScoreTableError(f'{file.name}: 未找到目标层及分量。')
    meta = _metadata(d, file)
    if 'model' not in d:
        label = _key(file.stem)
        matches = {mod for mod, names in _MODEL_NAMES.items() if any(a in label for a in names if len(a) >= 5)}
        if len(matches) == 1 and any(_which(c, _METRIC_NAMES) for c in d.columns):
            d['model'] = next(iter(matches))
    if 'model' in d:
        d['model'] = d.model.map(lambda v: _which(v, _MODEL_NAMES))
    bases = ['lead', 'level', 'component']
    if 'model' in d and 'metric' in d and 'value' in d:
        long = d.rename(columns={'value': 'score'}).copy()
        long['metric'] = long.metric.map(lambda v: _which(v, _METRIC_NAMES))
    else:
        pieces = []
        for col in d.columns:
            name = _key(col)
            if 'model' in d:
                metric = _which(name, _METRIC_NAMES)
                if metric:
                    pieces.append(d[bases + ['model']].assign(metric=metric, score=d[col]))
            else:
                if 'metric' in d and _which(name, _MODEL_NAMES):
                    pieces.append(d[bases].assign(model=_which(name, _MODEL_NAMES),
                        metric=d.metric.map(lambda v: _which(v, _METRIC_NAMES)), score=d[col]))
                    continue
                for model, aliases in _MODEL_NAMES.items():
                    for metric, tags in _METRIC_NAMES.items():
                        if any(name in (a+b, b+a) for a in aliases for b in tags):
                            pieces.append(d[bases].assign(model=model, metric=metric, score=d[col]))
        if not pieces:
            raise ScoreTableError(f'{file.name}: 没有识别到模型及绝对 R、NRMSE/RMSPE 列。')
        long = pd.concat(pieces, ignore_index=True)
    long = long.dropna(subset=['model','metric']).copy()
    if long.empty:
        raise ScoreTableError(f'{file.name}: 没有识别到三个模型的绝对指标行。')
    long['score'] = pd.to_numeric(long.score, errors='raise')
    if not np.isfinite(long.score).all():
        raise ScoreTableError(f'{file.name}: 指标包含缺失或无穷值。')
    return long[_IDENTIFIERS + ['metric', 'score']], meta


def _complete(frames, files):
    info = {}
    for _, metadata in frames:
        for key, value in metadata.items():
            if key in info and info[key] != value:
                raise ScoreTableError(f'汇总表间 {key} 不一致。')
            info[key] = value
    d = pd.concat([x for x, _ in frames], ignore_index=True)
    idx = _IDENTIFIERS + ['metric']
    if d.duplicated(idx).any():
        raise ScoreTableError('同一模型、时效、气压层、风分量和指标有重复行，无法确定唯一评分。')
    scores = d.pivot(index=_IDENTIFIERS, columns='metric', values='score')
    expected = pd.MultiIndex.from_product([_MODEL_NAMES, range(1,7), (850,925,1000), _COMPONENTS], names=_IDENTIFIERS)
    scores = scores.reindex(index=expected, columns=_SCORE_METRICS)
    if scores.isna().any().any():
        raise ScoreTableError(f'完整性 {int(scores.notna().sum().sum())}/324 个绝对评分。')
    if not scores.R.between(-1, 1).all() or (scores.RMSPE < 0).any():
        raise ScoreTableError('评分超出 Pearson R 或 RMSPE 的取值范围。')
    return scores.rename_axis(columns=None).reset_index(), files


def read_score_tables(root, files=None, filters=None):
    """Read an explicit set, or find a complete comparison table or a matching family of per-lead tables."""
    root, filters = Path(root), filters or {}
    if files:
        paths = [Path(p) for p in ([files] if isinstance(files, (str, Path)) else files)]
        return _complete([_one_table(p, filters) for p in paths], paths)
    if not root.is_dir():
        raise ScoreTableError(f'未找到评分目录 {root}。')
    ignored = re.compile(r'bootstrap|quantile|terrain|above.?ground|uncertainty|delta|difference|ranking|backup|archive|pilot|debug|temporary', re.I)
    candidate = [p for p in sorted(root.rglob('*.csv'))
                 if not ignored.search(p.relative_to(root).as_posix())
                 and re.search(r'score|metric|three.?models|compar|common|control|direct11', p.name, re.I)]
    valid, diagnostics = [], []
    groups = {}
    for path in candidate:
        try:
            parsed = _one_table(path, filters)
        except (ScoreTableError, ValueError, UnicodeError, OSError) as err:
            diagnostics.append(f'{path.name}: {err}')
            continue
        try:
            valid.append(_complete([parsed], [path]))
            continue
        except ScoreTableError as err:
            diagnostics.append(f'{path.name}: {err}')
        key = re.sub(r'(?i)(lead[_-]?|(?<![a-z])h)([1-6])(?=[_. /-]|$)', r'\1*', path.as_posix())
        groups.setdefault(key, []).append((parsed, path))
    for items in groups.values():
        if len(items) < 2:
            continue
        try:
            valid.append(_complete([pair[0] for pair in items], [pair[1] for pair in items]))
        except ScoreTableError as err:
            diagnostics.append(str(err))
    if not valid:
        partials = [record for family in groups.values() for record in family]
        if len(partials) > 1:
            try:
                valid.append(_complete([r[0] for r in partials], [r[1] for r in partials]))
            except ScoreTableError as err:
                diagnostics.append(str(err))
    if len(valid) == 1:
        return valid[0]
    if valid:
        chosen, paths = valid[0]
        if all(v[0].equals(chosen) for v in valid[1:]):
            return chosen, paths
        raise ScoreTableError('有多套数值不同的完整汇总。请在 TABLE_FILES 指定正式文件：\n'
                              + '\n'.join(str(v[1][0]) for v in valid))
    raise ScoreTableError('未找到覆盖全部 324 个分层绝对评分的唯一汇总。\n'
                          + '\n'.join(diagnostics[:12]))


In [ ]:
from pathlib import Path
import re
import numpy as np

def _raw_token(value):
    if isinstance(value, (bytes, np.bytes_)):
        value = value.decode("utf-8")
    return re.sub(r"[^a-z0-9]+", "", str(value).lower().replace("ω", "omega"))

def _raw_model(model):
    token = _raw_token(model)
    forms = {"direct6": "direct6", "direct6h": "direct6", "direct11": "direct11",
             "direct11h": "direct11", "twostage": "twostage", "twostageda": "twostage",
             "era5": "era5"}
    if token not in forms:
        raise ValueError(f"未知模型 {model!r}")
    return forms[token]

def _raw_component(component):
    aliases = {"u": ("u", "zonalwind", "ucomponentofwind", "eastwardwind"),
               "v": ("v", "meridionalwind", "vcomponentofwind", "northwardwind"),
               "w": ("w", "omega", "verticalvelocity", "pressureverticalvelocity",
                     "lagrangiantendencyofairpressure")}
    key = _raw_token(component)
    for c, names in aliases.items():
        if key in names:
            return c, names
    raise ValueError(f"未知风分量 {component!r}")

def raw_path(result_dir, model, kind, component, lead):
    key = _raw_model(model)
    c, _ = _raw_component(component)
    if kind not in ("predicty", "testy") or int(lead) not in range(1, 7):
        raise ValueError((kind, lead))
    root = Path(result_dir)
    if key == "direct11":
        return root / "direct11h_single_stage" / f"Direct11_{int(lead)}hour_{kind}.nc"
    if key not in ("direct6", "twostage"):
        raise ValueError("ERA5 文件用 raw_era5_path() 定位。")
    suffix = "_residual" if key == "twostage" else ""
    return root / f"da_transunet_{kind}_{c}_{int(lead)}_24year_changjiang{suffix}.nc"

def raw_era5_path(directory, component, level):
    c, _ = _raw_component(component)
    name = {"u": "U-component-of-wind", "v": "V-component-of-wind", "w": "Vertical-velocity"}[c]
    return Path(directory) / f"{name}-{int(level)}hpa.nc"

_RAW_AXES = {
    "time": ("targettime", "validtime", "verificationtime", "targettimestamp", "validtimestamp", "targettimes", "validtimes",
             "time", "times", "datetime", "date", "timestamp"),
    "latitude": ("latitude", "lat"), "longitude": ("longitude", "lon"),
    "level": ("level", "levels", "pressure", "pressurelevel", "pressurelevelhpa", "isobaricinhpa", "isobaricinpa", "plev"),
    "component": ("component", "components", "windcomponent", "variable", "variables", "var"),
    "channel": ("channel", "channels", "feature", "features", "target", "yfeature",
                "outputchannel", "outputchannels", "outputfeature", "outputfeatures", "outputs", "output", "outputvariable"),
}

def _raw_match_axis(da, role, overrides):
    if role in overrides:
        name = overrides[role]
        if name not in da.dims and name not in da.coords:
            raise ValueError(f"{role} 映射 {name!r} 不在 {da.name} 的维度或坐标中。")
        return name
    candidates = [name for name in (*da.coords, *da.dims) if _raw_token(name) in _RAW_AXES[role]]
    ordered = sorted(set(candidates), key=lambda n: _RAW_AXES[role].index(_raw_token(n)))
    if role == "time":
        for name in ordered:
            if name in da.coords and da[name].ndim == 1 and np.issubdtype(da[name].dtype, np.datetime64):
                return name
    if len(ordered) > 1:
        raise ValueError(f"{da.name} 有多个 {role} 坐标 {ordered}，请在 dims 中指定。")
    return ordered[0] if ordered else None

def _raw_hpa(values, units):
    values = np.asarray(values, dtype=float)
    unit = _raw_token(units)
    if unit in ("pa", "pascal", "pascals") or (not unit and np.all(values > 2000)):
        return values / 100
    if unit not in ("", "hpa", "hectopascal", "hectopascals", "mb", "mbar", "millibar", "millibars"):
        raise ValueError(f"无法识别的气压层单位 {units!r}")
    return values

_RAW_KIND_NAMES = {"predicty": ("predicty", "ypred", "prediction", "predictions", "pred", "forecast", "forecasts"),
                   "testy": ("testy", "ytrue", "truth", "reference", "target", "observation", "observed")}

def _raw_level_component_label(label, names, level):
    label = _raw_token(label)
    for tag in (*_RAW_KIND_NAMES["predicty"], *_RAW_KIND_NAMES["testy"]):
        if label.startswith(tag):
            label = label[len(tag):]
            break
        if label.endswith(tag):
            label = label[:-len(tag)]
            break
    label = label.replace("hpa", "").replace("mbar", "")
    return any(label in (f"{name}{level}", f"{level}{name}") for name in names)

def _raw_coord_dimension(da, coord):
    if coord in da.coords and da[coord].ndim == 1:
        return da[coord].dims[0]
    if coord in da.dims:
        return coord
    raise ValueError(f"{coord} 不是一维坐标。")

def raw_plane(ds, model, kind, component, level, *, variable=None, dims=None, channels=None, omega_factor=None):
    """Return (lazy time/latitude/longitude plane, unit multiplier). None = unresolved Direct11 ω unit.
    dims: canonical role -> real coordinate/dimension. channels: (u|v|w, hPa) -> packed channel index.
    For separate unlabeled axes channels may also map c -> component index and ("level", hPa) -> level index.
    Explicit variable can name a single component/level variable or a packed field.
    """
    key = _raw_model(model)
    c, names = _raw_component(component)
    level, dims = int(level), dict(dims or {})
    if kind not in ("predicty", "testy"):
        raise ValueError(kind)
    explicit_variable = variable is not None
    wrong = _RAW_KIND_NAMES["testy" if kind == "predicty" else "predicty"]
    vars_ = [v for v in ds.data_vars if not any(_raw_token(v).startswith(tag) or
                                                _raw_token(v).endswith(tag) for tag in wrong)]
    tags = _RAW_KIND_NAMES[kind]
    component_names = set(names) | {tag+n for tag in tags for n in names} | {n+tag for tag in tags for n in names}
    by_level = [v for v in vars_ if _raw_level_component_label(v, names, level)]
    by_component = [v for v in vars_ if _raw_token(v) in component_names]
    by_kind = [v for v in vars_ if _raw_token(v) in tags]
    if variable is None:
        picks = by_level or by_component or by_kind
        if not picks:
            picks = [v for v in vars_ if ds[v].ndim >= 4 and
                     (_raw_match_axis(ds[v], "channel", dims) or _raw_match_axis(ds[v], "component", dims))]
        if len(picks) != 1:
            raise ValueError(f"{model} {kind} {c}{level}: 可用变量 {list(ds.data_vars)}；请指定 variable。")
        variable = picks[0]
    if variable not in vars_:
        raise ValueError(f"{variable!r} 未出现在 {kind} 的候选变量 {vars_} 中。")
    original = ds[variable]
    da = original
    component_selected = variable in by_level + by_component or explicit_variable
    level_selected = variable in by_level or explicit_variable or (key == "era5" and variable in by_component)
    channel = _raw_match_axis(da, "channel", dims)
    comp_axis = _raw_match_axis(da, "component", dims)
    lev_axis = _raw_match_axis(da, "level", dims)
    if channel is None and comp_axis and lev_axis is None:
        channel, comp_axis = comp_axis, None
    if channel and lev_axis:
        if da[lev_axis].ndim in (0, 1) and _raw_coord_dimension(da, channel) != (
                da[lev_axis].dims[0] if da[lev_axis].ndim == 1 else None):
            if comp_axis is None and da.sizes[_raw_coord_dimension(da, channel)] == 3:
                comp_axis, channel = channel, None

    if comp_axis and lev_axis and comp_axis in da.coords and lev_axis in da.coords:
        cc, pp = da[comp_axis], da[lev_axis]
        if cc.ndim == pp.ndim == 1 and cc.dims == pp.dims:
            ix = np.flatnonzero(np.asarray([_raw_token(x) in names for x in cc.values]) &
                                np.isclose(_raw_hpa(pp.values, pp.attrs.get("units", "")), level))
            if len(ix) != 1:
                raise ValueError(f"{variable}: {c}{level} 联合坐标不唯一。")
            da = da.isel({cc.dims[0]: int(ix[0])}, drop=True)
            channel = comp_axis = lev_axis = None
            component_selected = level_selected = True
    if channel:
        dim = _raw_coord_dimension(da, channel)
        if channels is not None and (c, level) in channels:
            index = channels[c, level]
            if not isinstance(index, (int, np.integer)) or not (0 <= index < da.sizes[dim]):
                raise ValueError(f"{(c, level)} 通道位置 {index!r} 越界。")
        else:
            labels = da[channel].values if channel in da.coords else []
            hits = [i for i, label in enumerate(labels) if _raw_level_component_label(label, names, level)]
            if len(hits) != 1:
                raise ValueError(f"{variable}: 无法从 {channel}={list(labels)} 选 {c}{level}，"
                                 "请按正式导出代码提供 channels[(分量, hPa)]。")
            index = hits[0]
        da = da.isel({dim: int(index)}, drop=True)
        level_selected = component_selected = True
    if comp_axis:
        coord = da[comp_axis]
        labels = np.atleast_1d(coord.values)
        hits = [i for i, x in enumerate(labels) if _raw_token(x) in names]
        if not hits and channels is not None and c in channels:
            i = channels[c]
            if isinstance(i, (int, np.integer)) and 0 <= i < len(labels):
                hits = [int(i)]
        if len(hits) != 1:
            raise ValueError(f"{variable}: {comp_axis}={list(labels)}，无法唯一识别分量 {c}。")
        if coord.ndim:
            da = da.isel({coord.dims[0]: hits[0]}, drop=True)
        component_selected = True
    if lev_axis:
        coord = da[lev_axis]
        labels = np.atleast_1d(_raw_hpa(coord.values, coord.attrs.get("units", "")))
        hits = np.flatnonzero(np.isclose(labels, level, rtol=0, atol=1e-6))
        if len(hits) == 0 and channels is not None and ("level", level) in channels:
            i = channels["level", level]
            if isinstance(i, (int, np.integer)) and 0 <= i < len(labels):
                hits = [int(i)]
        if len(hits) != 1:
            raise ValueError(f"{variable}: {labels.tolist()} 中未找到唯一的 {level} hPa。")
        if coord.ndim:
            da = da.isel({coord.dims[0]: int(hits[0])}, drop=True)
        level_selected = True
    if not (component_selected and level_selected):
        raise ValueError(f"{variable}: 缺少可核对的分量/层次坐标，当前 {c}{level}。")
    for role in ("time", "latitude", "longitude"):
        coord = _raw_match_axis(da, role, dims)
        if not coord or coord not in da.coords or da[coord].ndim != 1:
            raise ValueError(f"{variable}: 缺少可核对的一维 {role} 坐标；请指定 dims。")
        dim = da[coord].dims[0]
        if dim != coord:
            da = da.swap_dims({dim: coord})
        if coord != role:
            if role in da.coords and role not in da.dims:
                da = da.drop_vars(role)
            da = da.rename({coord: role})
    for dim in set(da.dims) - {"time", "latitude", "longitude"}:
        if da.sizes[dim] != 1:
            raise ValueError(f"{variable}: 存在未处理维度 {dim}，长度 {da.sizes[dim]}。")
        da = da.isel({dim: 0}, drop=True)
    clock = _raw_token(da.time.attrs.get("standard_name", ""))
    if clock in ("forecastreferencetime", "forecastinitializationtime", "forecastissuetime"):
        raise ValueError(f"{variable}: 当前为起报时刻，请用 dims['time'] 显式选择目标有效时刻。")
    tt = da.time.values
    if not np.issubdtype(tt.dtype, np.datetime64):
        raise ValueError(f"{variable}: 时间坐标未解析为日期，请用 decode_times=True 打开 NC。")
    tt = tt.astype("datetime64[ns]")
    arr = tt.astype("int64")
    hour = 3_600_000_000_000
    offset = (arr % hour + hour // 2) % hour - hour // 2
    if np.isnat(tt).any() or np.any(np.abs(offset) > 1000):
        raise ValueError(f"{variable}: 存在缺失或偏离整点超过 1 µs 的目标时间。")
    tt = (arr-offset).astype("datetime64[ns]")
    if len(np.unique(tt)) != len(tt):
        raise ValueError(f"{variable}: 整点归一后存在重复时次。")
    da = da.assign_coords(time=tt)
    for axis in ("latitude", "longitude", "time"):
        vals = np.asarray(da[axis].values)
        if not np.isfinite(vals).all() or len(np.unique(vals)) != len(vals):
            raise ValueError(f"{variable}: {axis} 含重复或非法坐标。")
        if len(vals) > 1 and not np.all(vals[1:] > vals[:-1]):
            order = slice(None, None, -1) if np.all(vals[1:] < vals[:-1]) else np.argsort(vals)
            da = da.isel({axis: order})
    scale = 1.0
    if c == "w":
        if omega_factor is not None:
            scale = float(omega_factor)
            if not np.isfinite(scale) or scale == 0:
                raise ValueError(f"无效的 omega_factor={omega_factor}")
        elif key in ("direct6", "twostage"):
            scale = -10.0
        elif key != "era5":
            unit = _raw_token(original.attrs.get("units", ""))
            scale = 1.0 if unit in ("pas", "pas1", "pasecond1", "pascalsecond1",
                                    "pascalspersecond", "pascalpersecond", "papersecond") else None
    return da.transpose("time", "latitude", "longitude"), scale


In [ ]:
from contextlib import ExitStack
from pathlib import Path
import numpy as np
import pandas as pd


def _compact_index(indices):
    """Prefer a contiguous on-disk slice, including a reversed latitude axis."""
    if len(indices) == 1:
        return slice(int(indices[0]), int(indices[0]) + 1)
    step = np.diff(indices)
    if np.all(step == 1):
        return slice(int(indices[0]), int(indices[-1]) + 1)
    if np.all(step == -1):
        return slice(int(indices[0]), int(indices[-1]) - 1 if indices[-1] else None, -1)
    return indices


def _prepare_plane(da, times, lat, lon, *, truth=False):
    if set(da.dims) != {"time", "latitude", "longitude"}:
        raise ValueError(f"{da.name}: 期待 time/latitude/longitude，实际 {da.dims}")
    da = da.transpose("time", "latitude", "longitude")
    if not np.issubdtype(da.time.dtype, np.datetime64):
        raise ValueError(f"{da.name}: 需要可核对的公历目标时刻，实际 {da.time.dtype}")
    source = pd.DatetimeIndex(da.time.values)
    hour = source.round("h")
    if source.hasnans or hour.has_duplicates or (np.abs(source - hour) > pd.Timedelta(milliseconds=1)).any():
        raise ValueError(f"{da.name}: 目标时刻含缺失、重复或非整点记录")
    tids = hour.get_indexer(times)
    if (tids < 0).any():
        missing = times[tids < 0]
        raise ValueError(f"{da.name}: 缺少 {len(missing)} 个共同目标时刻，例如 {missing[0]}")
    indexers = {}
    for dim, expected in (("latitude", lat), ("longitude", lon)):
        actual = np.asarray(da[dim], dtype=float)
        if not np.isfinite(actual).all() or (not truth and len(actual) != len(expected)):
            raise ValueError(f"{da.name}: {dim} 覆盖与正式计算网格不一致")
        order = np.argsort(actual)
        lookup = pd.Index(actual[order])
        if not lookup.is_unique:
            raise ValueError(f"{da.name}: {dim} 含重复坐标")
        idx = lookup.get_indexer(expected, method="nearest", tolerance=1e-6)
        if (idx < 0).any() or len(np.unique(idx)) != len(expected):
            raise ValueError(f"{da.name}: {dim} 网格不一致，无法逐格比较")
        indexers[dim] = _compact_index(order[idx])
    return da.isel(indexers), tids


def _block_values(prepared, start, end, factor=1.0):
    da, tids = prepared
    a = np.asarray(da.isel(time=_compact_index(tids[start:end])).values, dtype=np.float64)
    return a if factor == 1.0 else a * factor


def _paired_w_scale(reference, target, n, documented):
    candidates = (1.0, -10.0) if documented is None else (float(documented),)
    count = 0
    for i in np.linspace(0, n - 1, min(5, n), dtype=int):
        y = _block_values(reference, i, i + 1)
        a = _block_values(target, i, i + 1)
        valid = np.isfinite(y) & np.isfinite(a) & (np.abs(y) > 1e-5)
        count += np.count_nonzero(valid)
        if valid.any():
            candidates = tuple(k for k in candidates if np.allclose(a[valid] * k, y[valid], rtol=1e-4, atol=3e-6))
    if not count or len(candidates) != 1:
        raise ValueError("Direct11 的 ω testy 与原始 ERA5 抽查未能确定唯一物理单位。请核对发布记录并指定 omega_factor。")
    return candidates[0]


def _regional_scores(reference, predictions, times, lat, lon, block_hours):
    shape = (len(lat), len(lon))
    n = np.zeros(shape, dtype=np.int64)
    mean = np.zeros((4, *shape))
    m2 = np.zeros_like(mean)
    cross = np.zeros((3, *shape))
    sse = np.zeros_like(cross)
    lo, hi = np.full_like(mean, np.inf), np.full_like(mean, -np.inf)
    fields = [reference, *predictions]
    for start in range(0, len(times), block_hours):
        end = min(start + block_hours, len(times))
        vals = [_block_values(plane, start, end, scale) for plane, scale in fields]
        ok = np.logical_and.reduce([np.isfinite(a) for a in vals])
        b = ok.sum(axis=0)
        total = n + b
        delta = []
        centered = []
        for j, a in enumerate(vals):
            np.minimum(lo[j], np.min(a, where=ok, initial=np.inf, axis=0), out=lo[j])
            np.maximum(hi[j], np.max(a, where=ok, initial=-np.inf, axis=0), out=hi[j])
            avg = np.sum(a, where=ok, axis=0) / np.maximum(b, 1)
            d = avg - mean[j]
            delta.append(d)
            v = np.where(ok, a - avg, 0.0)
            centered.append(v)
            m2[j] += np.einsum("ijk,ijk->jk", v, v) + d*d*n*b/np.maximum(total, 1)
            mean[j] += d*b/np.maximum(total, 1)
        for j, a in enumerate(vals[1:]):
            cross[j] += (np.einsum("ijk,ijk->jk", centered[0], centered[j+1])
                         + delta[0]*delta[j+1]*n*b/np.maximum(total, 1))
            diff = np.subtract(a, vals[0], where=ok, out=np.zeros_like(a))
            sse[j] += np.einsum("ijk,ijk->jk", diff, diff)
        n = total
    valid_err = (n >= 3) & (hi[0] > lo[0])
    valid_r = valid_err & np.all((m2 > 0) & (hi > lo), axis=0)
    if not valid_r.any() or not valid_err.any():
        raise ValueError("共同支持域为空，请核对原始 NC 和坐标。")
    r = np.clip(cross[:, valid_r] / np.sqrt(m2[0, valid_r] * m2[1:, valid_r]), -1, 1)
    err = np.sqrt(sse[:, valid_err] / n[valid_err]) / (hi[0, valid_err] - lo[0, valid_err])
    return r.mean(axis=1), err.mean(axis=1)


def read_raw_scores(result_dir, overrides=None, block_hours=168, schema=None,
                    era5_dir=DATA):
    """Recompute grid-time scores from final NC and common original ERA5 truth."""
    import xarray as xr
    if not isinstance(block_hours, (int, np.integer)) or block_hours < 1:
        raise ValueError("RAW_BLOCK_HOURS 应为正整数。")
    overrides, schema = overrides or {}, schema or {}
    models = ("Direct6", "Direct11", "Two-stage")
    times = pd.date_range("2019-03-15 02:00", "2023-12-31 23:00", freq="h")
    lat, lon = np.arange(24., 36.125, .25), np.arange(108., 126.125, .25)
    used, records = set(), []
    for lead in range(1, 7):
        with ExitStack() as stack:
            opened = {}
            def get(model, kind, comp, level):
                key = (model, kind, comp, level if model == "ERA5" else lead)
                if key in overrides:
                    p = Path(overrides[key])
                elif model == "ERA5":
                    p = Path(raw_era5_path(era5_dir, comp, level))
                else:
                    p = Path(raw_path(result_dir, model, kind, comp, lead))
                if not p.is_file():
                    raise FileNotFoundError(f"缺少 {key} 文件 {p}。请核对已验收路径或填写 RAW_PATHS。")
                if p not in opened:
                    opened[p] = stack.enter_context(xr.open_dataset(p, cache=False))
                kwargs = schema.get((model, kind, comp), schema.get((model, kind), {}))
                plane, factor = raw_plane(opened[p], model, kind, comp, level, **kwargs)
                used.add(p)
                return _prepare_plane(plane, times, lat, lon, truth=model == "ERA5"), factor, p
            for comp in ("u", "v", "omega"):
                for level in (850, 925, 1000):
                    ref, ref_scale, _ = get("ERA5", "testy", comp, level)
                    if ref_scale != 1.0:
                        raise ValueError("请核对原始 ERA5 的物理单位。")
                    predictions = []
                    for model in models:
                        field, factor, path = get(model, "predicty", comp, level)
                        if factor is None and model == "Direct11" and comp == "omega":
                            gt, target_scale, target_path = get(model, "testy", comp, level)
                            if path.name.lower().replace("predicty", "testy") != target_path.name.lower() or path.parent != target_path.parent:
                                raise ValueError("请根据发布记录为自定义 ω 预测文件明确指定 omega_factor。")
                            factor = _paired_w_scale(ref, gt, len(times), target_scale)
                            print(f"  {lead} h {level} hPa：Direct11 同批导出 ω 抽查，乘数 {factor:g}")
                        if factor is None or not np.isfinite(factor) or factor == 0:
                            raise ValueError(f"请核对 {path} 的物理单位并指定换算系数。")
                        predictions.append((field, factor))
                    rs, es = _regional_scores((ref, ref_scale), predictions, times, lat, lon, block_hours)
                    for model, r, e in zip(models, rs, es):
                        records.append(dict(model=model, lead=lead, level=level, component=comp, R=r, RMSPE=e))
        print(f"完成 {lead}/6 个时效的原始 NC 评分。", flush=True)
    return pd.DataFrame(records), sorted(used)


In [ ]:
if SOURCE not in {"auto", "tables", "netcdf"}:
    raise ValueError("SOURCE 应为 auto、tables 或 netcdf。")

scores = None
source_paths = []
if SOURCE in {"auto", "tables"}:
    try:
        candidate, selected = read_score_tables(SCORE_DIR, files=TABLE_FILES, filters=TABLE_FILTERS)
        scores, source_paths = candidate, selected
        print("已使用分层绝对评分表：", *[str(p) for p in source_paths], sep="\n  ")
    except (ScoreTableError, FileNotFoundError) as error:
        if SOURCE == "tables" or TABLE_FILES:
            raise
        print("已有表未能提供整套可核验的分层绝对评分，将改读预测 NC。")
        print(str(error)[:1600])

from_raw = scores is None
if from_raw:
    scores, source_paths = read_raw_scores(
        RESULT_DIR, overrides=RAW_PATHS, block_hours=BLOCK_HOURS, schema=RAW_SCHEMA, era5_dir=ERA5_DIR
    )
    print(f"分块计算完成。使用了 {len(source_paths)} 个源文件，路径保存在 source_paths。")

keys = ["model", "lead", "level", "component"]
wanted = pd.MultiIndex.from_product(
    [["Direct6", "Direct11", "Two-stage"], range(1, 7), [850, 925, 1000], ["u", "v", "omega"]],
    names=keys
)
if len(scores) != len(wanted) or not pd.MultiIndex.from_frame(scores[keys]).sort_values().equals(wanted.sort_values()):
    raise ValueError("评分表必须恰好覆盖 3 个模型 × 6 个时效 × 3 个气压层 × 3 个分量。")
values = scores[["R", "RMSPE"]]
if not np.isfinite(values.to_numpy()).all() or not scores.R.between(-1, 1).all() or (scores.RMSPE < 0).any():
    raise ValueError("完整评分中有缺测或取值越界，请检查源文件。")
print(f"已核对 {len(scores)} 组比较、{values.size} 个指标值。")


if from_raw:
    import os
    from tempfile import TemporaryDirectory
    RAW_SCORE_FILE.parent.mkdir(parents=True, exist_ok=True)
    exported = scores.assign(n_common_targets=42070,
                             valid_time_start='2019-03-15 02:00:00+00:00',
                             valid_time_end='2023-12-31 23:00:00+00:00',
                             truth_source='ERA5', evaluation_domain='full domain')
    with TemporaryDirectory(dir=RAW_SCORE_FILE.parent) as stage:
        temporary = Path(stage) / RAW_SCORE_FILE.name
        exported.to_csv(temporary, index=False)
        os.replace(temporary, RAW_SCORE_FILE)


In [ ]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, ScalarFormatter
from PIL import Image


def plot_comparison(scores):
    """Draw six forecast leads per model in each metric–level–component panel."""
    levels = (850, 925, 1000)
    components = ("u", "v", "omega")
    metrics = ("R", "RMSPE")
    leads = list(range(1, 7))
    titles = ("Zonal wind, $u$", "Meridional wind, $v$", "Pressure vertical velocity, $\\omega$")
    styles = {
        "Direct6": ("Direct 6 h", "#3167A2", "--", "o", 6.4, "none"),
        "Direct11": ("Direct 11 h", "#B85A1D", "-.", "s", 4.5, "none"),
        "Two-stage": ("Two-stage", "#176E63", "-", "D", 2.6, "#176E63"),
    }
    data = scores.set_index(["level", "component", "model", "lead"]).sort_index()
    with mpl.rc_context({
        "font.family": "sans-serif", "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 9, "axes.unicode_minus": True, "mathtext.fontset": "dejavusans",
    }):
        fig = plt.figure(figsize=(7.7, 10.2), facecolor="white")
        grid = fig.add_gridspec(3, 3, left=0.141, right=0.986, bottom=0.094, top=0.942,
                                hspace=0.32, wspace=0.32)
        handles = []
        for group, level in enumerate(levels):
            for col, comp in enumerate(components):
                pair = grid[group, col].subgridspec(2, 1, hspace=0.24)
                pair_axes = []
                for k, metric in enumerate(metrics):
                    ax = fig.add_subplot(pair[k, 0])
                    pair_axes.append(ax)
                    ax.set_axisbelow(True)
                    ax.grid(axis="y", color="#DCE1E4", linewidth=0.55)
                    for side in ("top", "right"):
                        ax.spines[side].set_visible(False)
                    for side in ("left", "bottom"):
                        ax.spines[side].set_color("#7D878E")
                        ax.spines[side].set_linewidth(0.65)
                    ax.tick_params(length=2.6, width=0.65, pad=2.2, labelsize=8.5)
                    for model, (label, color, line, marker, size, fill) in styles.items():
                        trace = data.loc[(level, comp, model), metric].loc[leads]
                        handle, = ax.plot(leads, trace.to_numpy(dtype=float), color=color,
                                          linestyle=line, linewidth=1.25, marker=marker,
                                          markersize=size, markerfacecolor=fill,
                                          markeredgewidth=1.0, label=label)
                        if group == col == k == 0:
                            handles.append(handle)
                    ax.set_xlim(0.74, 6.26)
                    ax.set_xticks(leads)
                    ax.tick_params(axis="x", labelbottom=(k == 1))
                    ax.margins(y=0.11)
                    ax.yaxis.set_major_locator(MaxNLocator(nbins=4, min_n_ticks=3,
                                                           steps=[1, 2, 2.5, 5, 10]))
                    formatter = ScalarFormatter(useOffset=False, useMathText=True)
                    formatter.set_powerlimits((-3, 4))
                    ax.yaxis.set_major_formatter(formatter)
                    ax.yaxis.get_offset_text().set_fontsize(7.5)
                    letter = chr(ord("a") + 3 * (2 * group + k) + col)
                    ax.text(1.0, 1.015, f"({letter})", transform=ax.transAxes,
                            ha="right", va="bottom", fontsize=8.6)
                    if col == 0:
                        ax.set_ylabel(metric, fontsize=9.8, rotation=90)
                        ax.yaxis.set_label_coords(-0.28, 0.5)
                if group == 0:
                    box = pair_axes[0].get_position()
                    fig.text((box.x0 + box.x1) / 2, 0.978, titles[col],
                             ha="center", va="center", fontsize=9.8)
                if col == 0:
                    ymid = (pair_axes[0].get_position().y1 + pair_axes[1].get_position().y0) / 2
                    fig.text(0.016, ymid, f"{level} hPa", rotation=90, ha="center",
                             va="center", fontsize=11.0, fontweight="bold")
        fig.text(0.563, 0.051, "Forecast lead (h)", ha="center", va="center", fontsize=10.5)
        fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.563, 0.006),
                   frameon=False, ncol=3, fontsize=9.8, handlelength=3.2,
                   handletextpad=0.6, columnspacing=2.2)
    return fig


def save_comparison(fig, directory, dpi=700):
    """Stage a 700-dpi PNG and create the LZW TIFF from its decoded RGB pixels."""
    if dpi < 600:
        raise ValueError("Use at least 600 dpi for publication.")
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / "FigS24.png"
    tiff = directory / "FigS24.tiff"
    with TemporaryDirectory(dir=directory) as tmp:
        staged_png = Path(tmp) / png.name
        staged_tiff = Path(tmp) / tiff.name
        fig.savefig(staged_png, dpi=dpi, format="png", facecolor="white")
        with Image.open(staged_png) as im:
            expected = im.size
            with im.convert("RGB") as rgb:
                rgb.save(staged_tiff, format="TIFF", compression="tiff_lzw", dpi=(dpi, dpi))
        with Image.open(staged_tiff) as im:
            if im.size != expected:
                raise ValueError("PNG and TIFF dimensions differ.")
            im.verify()
        os.replace(staged_png, png)
        os.replace(staged_tiff, tiff)
    return png, tiff


In [ ]:
fig = plot_comparison(scores)
png_path, tiff_path = save_comparison(fig, FIGURE_DIR, dpi=DPI)
print(f"{DPI} dpi PNG　{png_path}")
print(f"{DPI} dpi TIFF　{tiff_path}")
plt.show()
